# CreditParity – Step 8
**BUS/FIN 241A · Final Team Project · Profitable, Fair Lending Decisions**

**Step 8:** Save the final model, the decision policy and the comparison model

## Step 8 · Saving the Model

**Goal.** Package everything the dashboard (Step 9) needs in `model/`, so the app never has to retrain:

| File | Content |
|---|---|
| `creditparity_bundle.joblib` | The **final model (d)** and the **comparison model (a)** as complete sklearn pipelines (preprocessing + logistic regression), the decision policy (zone boundaries, 80% precision target, 5:1 costs) and the data needed for exact SHAP explanations |
| `policy.json` | The same decision policy in a human-readable form |
| `evaluation_data.csv` | Default probabilities of both models for the 4,000 training applications (out-of-fold) and the 1,000 test applications, with the true outcome and the sensitive attributes. The dashboard's bank view uses it to recompute zones, cost, recall and fairness live when the sliders move. |

**Which data are the models trained on?** On the 4,000 training applications, exactly as in Steps 5–7. This keeps the test set untouched, so we can verify below that the saved files reproduce the reported test results. (A production model would usually be refit on all data before deployment; we keep the training-only version so that every number in the dashboard can be traced back to the notebooks.)

**Pipeline input.** Both pipelines accept a complete application with all 20 fields. Model (d)'s preprocessing simply ignores `Sex` and `ForeignWorker`, so the same input table works for both models.

In [1]:
# Libraries
import json
import os
import platform
import subprocess
import sys

import joblib
import numpy as np
import pandas as pd
import sklearn
import shap          # only used to cross-check the stored SHAP data

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42
MODEL_DIR = "../model"
os.makedirs(MODEL_DIR, exist_ok=True)

In [2]:
# Load data
# In the repo the file lives in data/. In Google Colab, upload the CSV via the
# folder icon on the left and set DATA_PATH = "german_credit_data_biased_training.csv"
DATA_PATH = "../data/german_credit_data_biased_training.csv"
df = pd.read_csv(DATA_PATH)
df["y"] = (df["Risk"] == "Risk").astype(int)

X = df.drop(columns=["Risk", "y"])
y = df["y"]
categorical = X.select_dtypes(include=["object", "string"]).columns.tolist()
numeric = X.select_dtypes(include="number").columns.tolist()

# Same stratified 80/20 split as in Steps 3 to 7
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

### a) Train the final and the comparison model

Same definitions as in Steps 5–7: model (d) uses reweighing on age group × sex × foreign worker status and all features except `Sex` and `ForeignWorker`; model (a) uses all features without reweighing.

In [3]:
ALL_FEATURES = X.columns.tolist()
FAIR_FEATURES = [c for c in ALL_FEATURES if c not in ["Sex", "ForeignWorker"]]

def make_model(features):
    return Pipeline([
        ("prep", ColumnTransformer([
            ("cat", OneHotEncoder(handle_unknown="ignore"), [c for c in categorical if c in features]),
            ("num", StandardScaler(), [c for c in numeric if c in features])])),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ])

def age_group(data):
    return pd.cut(data["Age"], [18, 25, 35, 50, 100],
                  labels=["19-25", "26-35", "36-50", "51+"]).astype(str)

def reweighing_weights(data, y_true):
    """Kamiran & Calders reweighing on age group x sex x foreign worker status."""
    group = age_group(data) + "|" + data["Sex"] + "|" + data["ForeignWorker"]
    w = pd.Series(1.0, index=data.index)
    for g in group.unique():
        for label in [0, 1]:
            m = (group == g) & (y_true == label)
            if m.any():
                w[m] = (group == g).mean() * (y_true == label).mean() / m.mean()
    return w.to_numpy()

SPECS = {"d": (FAIR_FEATURES, True), "a": (ALL_FEATURES, False)}

def fit_model(data, y_true, key):
    features, reweigh = SPECS[key]
    model = make_model(features)
    if reweigh:
        return model.fit(data, y_true, model__sample_weight=reweighing_weights(data, y_true))
    return model.fit(data, y_true)

models = {key: fit_model(X_train, y_train, key) for key in SPECS}
for key, m in models.items():
    print(f"Model ({key}): {len(m.named_steps['prep'].get_feature_names_out())} model columns, "
          f"test ROC AUC {roc_auc_score(y_test, m.predict_proba(X_test)[:, 1]):.3f}")

Model (d): 56 model columns, test ROC AUC 0.811
Model (a): 60 model columns, test ROC AUC 0.867


### b) Decision policy

The thresholds are recomputed here with exactly the rules of Step 5 (on out-of-fold predictions of the training set), so the saved values cannot drift from the notebook. We also store the upper boundary for other precision targets, which the dashboard slider uses.

In [4]:
COST_FN, COST_FP = 5, 1
TARGET_PRECISION = 0.80
THRESHOLDS = np.round(np.arange(0.01, 1.00, 0.01), 2)

def cost(p, y_true, t):
    return int(COST_FN * ((p < t) & (y_true == 1)).sum() + COST_FP * ((p >= t) & (y_true == 0)).sum())

def cost_optimal_threshold(p, y_true):
    return float(THRESHOLDS[int(np.argmin([cost(p, y_true, t) for t in THRESHOLDS]))])

def upper_for_precision(p, y_true, lower, target):
    for t in THRESHOLDS[THRESHOLDS >= lower]:
        red = p >= t
        if red.any() and y_true[red].mean() >= target:
            return float(t)
    return 1.0

# Out-of-fold predictions on the training set (same folds as Steps 4-7)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
y_tr = y_train.to_numpy()
p_oof = {key: np.zeros(len(y_tr)) for key in SPECS}
for tr, va in cv.split(X_train, y_train):
    for key in SPECS:
        p_oof[key][va] = fit_model(X_train.iloc[tr], y_train.iloc[tr], key).predict_proba(X_train.iloc[va])[:, 1]

TARGETS = [0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
policy = {"cost_fn": COST_FN, "cost_fp": COST_FP, "target_precision": TARGET_PRECISION, "models": {}}
for key in SPECS:
    lower = cost_optimal_threshold(p_oof[key], y_tr)
    policy["models"][key] = {
        "lower_boundary": lower,
        "upper_boundary": upper_for_precision(p_oof[key], y_tr, lower, TARGET_PRECISION),
        "upper_boundary_by_target": {f"{t:.2f}": upper_for_precision(p_oof[key], y_tr, lower, t)
                                     for t in TARGETS},
    }
print(json.dumps(policy, indent=2))

{
  "cost_fn": 5,
  "cost_fp": 1,
  "target_precision": 0.8,
  "models": {
    "d": {
      "lower_boundary": 0.15,
      "upper_boundary": 0.71,
      "upper_boundary_by_target": {
        "0.70": 0.53,
        "0.75": 0.62,
        "0.80": 0.71,
        "0.85": 0.77,
        "0.90": 0.82,
        "0.95": 0.89
      }
    },
    "a": {
      "lower_boundary": 0.11,
      "upper_boundary": 0.58,
      "upper_boundary_by_target": {
        "0.70": 0.47,
        "0.75": 0.52,
        "0.80": 0.58,
        "0.85": 0.66,
        "0.90": 0.73,
        "0.95": 0.86
      }
    }
  }
}


### c) Data for exact SHAP explanations

For a logistic regression, the exact SHAP value of a model column is *coefficient × (value − average value in the training data)* (Step 7). We store the average of each model column, so the dashboard can compute explanations with a few lines of numpy, and we check that this matches the `shap` library.

In [5]:
def dense(m):
    return m.toarray() if hasattr(m, "toarray") else np.asarray(m)

def original_feature(column):
    name = column.split("__", 1)[1]
    return next(f for f in ALL_FEATURES if name == f or name.startswith(f + "_"))

explain_data = {}
for key, m in models.items():
    prep = m.named_steps["prep"]
    columns = list(prep.get_feature_names_out())
    explain_data[key] = {
        "columns": columns,
        "feature_of_column": [original_feature(c) for c in columns],
        "background_mean": dense(prep.transform(X_train)).mean(axis=0),
    }

def shap_values(model, info, data):
    """Exact linear SHAP values per original feature (log-odds) and the base value."""
    Z = dense(model.named_steps["prep"].transform(data))
    coef = model.named_steps["model"].coef_[0]
    phi = pd.DataFrame(coef * (Z - info["background_mean"]), columns=info["columns"], index=data.index)
    base = float(model.named_steps["model"].intercept_[0] + coef @ info["background_mean"])
    return phi.T.groupby(info["feature_of_column"], sort=False).sum().T, base

# Cross-check against the shap library (all training applications as background)
m, info = models["d"], explain_data["d"]
Z_train = dense(m.named_steps["prep"].transform(X_train))
lib = shap.LinearExplainer(m.named_steps["model"],
                           shap.maskers.Independent(Z_train, max_samples=len(Z_train)))
lib_values = lib(dense(m.named_steps["prep"].transform(X_test))).values
own_values = m.named_steps["model"].coef_[0] * (
    dense(m.named_steps["prep"].transform(X_test)) - info["background_mean"])
print("Max. difference to the shap library:", float(np.abs(lib_values - own_values).max()))
phi, base = shap_values(m, info, X_test)
print("Max. difference base + sum(SHAP) vs. model log-odds:",
      float(np.abs(base + phi.sum(axis=1) - m.decision_function(X_test)).max()))

Max. difference to the shap library: 0.0
Max. difference base + sum(SHAP) vs. model log-odds: 2.6645352591003757e-15


### d) Save

In [6]:
bundle = {
    "final_model": models["d"],
    "comparison_model": models["a"],
    "policy": policy,
    "explain_data": explain_data,
    "metadata": {
        "final_model_name": "(d) Fair model: reweighing; Age as input, Sex and ForeignWorker not used",
        "comparison_model_name": "(a) All features, no reweighing",
        "input_features": ALL_FEATURES,
        "features_used_by_final_model": FAIR_FEATURES,
        "categorical_features": categorical,
        "numeric_features": numeric,
        "category_values": {c: sorted(X[c].unique().tolist()) for c in categorical},
        "numeric_ranges": {c: [int(X[c].min()), int(X[c].max())] for c in numeric},
        "trained_on": "4,000 training applications (stratified 80/20 split, random_state=42)",
        "sklearn_version": sklearn.__version__,
        "python_version": platform.python_version(),
    },
}
joblib.dump(bundle, f"{MODEL_DIR}/creditparity_bundle.joblib")

with open(f"{MODEL_DIR}/policy.json", "w") as f:
    json.dump({**policy, "final_model": bundle["metadata"]["final_model_name"],
               "comparison_model": bundle["metadata"]["comparison_model_name"]}, f, indent=2)

# Evaluation data for the dashboard's bank view
evaluation = pd.concat([
    X_train[["Sex", "Age", "ForeignWorker"]].assign(
        split="train_oof", y=y_tr, pd_final=p_oof["d"], pd_comparison=p_oof["a"]),
    X_test[["Sex", "Age", "ForeignWorker"]].assign(
        split="test", y=y_test.to_numpy(),
        pd_final=models["d"].predict_proba(X_test)[:, 1],
        pd_comparison=models["a"].predict_proba(X_test)[:, 1]),
])
evaluation["age_group"] = age_group(evaluation)
evaluation.round(6).to_csv(f"{MODEL_DIR}/evaluation_data.csv", index_label="application_id")

for f in sorted(os.listdir(MODEL_DIR)):
    print(f"{f:32s} {os.path.getsize(os.path.join(MODEL_DIR, f)) / 1024:8.1f} KB")

creditparity_bundle.joblib           19.8 KB
evaluation_data.csv                 254.5 KB
policy.json                           0.7 KB


### e) Verification: reload the saved files and reproduce the test results

We reload the files **in a separate Python process**, which knows nothing about the objects in this notebook, and recompute the test-set results. They must match Steps 5 and 6 exactly.

In [7]:
VERIFY_SCRIPT = r'''
import json, sys
import joblib, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

bundle = joblib.load("../model/creditparity_bundle.joblib")
policy = json.load(open("../model/policy.json"))
evaluation = pd.read_csv("../model/evaluation_data.csv", index_col="application_id")

df = pd.read_csv("../data/german_credit_data_biased_training.csv")
y = (df["Risk"] == "Risk").astype(int)
X = df.drop(columns=["Risk"])
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
yt = y_test.to_numpy()
age = pd.cut(X_test["Age"], [18, 25, 35, 50, 100], labels=["19-25", "26-35", "36-50", "51+"]).astype(str)

def evaluate(model, pol):
    p = model.predict_proba(X_test)[:, 1]
    lo, up = pol["lower_boundary"], pol["upper_boundary"]
    zone = np.where(p < lo, "G", np.where(p >= up, "R", "Y"))
    fpr = lambda g: pd.Series((p >= lo) & (yt == 0)).groupby(g.to_numpy()).sum() / pd.Series(yt == 0).groupby(g.to_numpy()).sum()
    good51 = (age.to_numpy() == "51+") & (yt == 0)
    return p, {
        "lower": lo, "upper": up,
        "cost": int(5 * ((p < lo) & (yt == 1)).sum() + ((p >= lo) & (yt == 0)).sum()),
        "recall": round(((p >= lo) & (yt == 1)).sum() / yt.sum(), 3),
        "green": round((zone == "G").mean(), 3), "yellow": round((zone == "Y").mean(), 3),
        "red": round((zone == "R").mean(), 3),
        "red_precision": round(yt[zone == "R"].mean(), 3),
        "good_auto_declined": int(((zone == "R") & (yt == 0)).sum()),
        "defaults_auto_approved": int(((zone == "G") & (yt == 1)).sum()),
        "eod_age": round(float(fpr(age).max() - fpr(age).min()), 3),
        "eod_sex": round(float(fpr(X_test["Sex"]).max() - fpr(X_test["Sex"]).min()), 3),
        "good_51plus_auto_declined": f"{int((good51 & (zone == 'R')).sum())} of {int(good51.sum())}",
    }

p_d, res_d = evaluate(bundle["final_model"], bundle["policy"]["models"]["d"])
p_a, res_a = evaluate(bundle["comparison_model"], bundle["policy"]["models"]["a"])
ev = evaluation[evaluation["split"] == "test"]
print(json.dumps({
    "final_model_d": res_d,
    "comparison_model_a": res_a,
    "policy_json_matches_bundle": policy["models"] == bundle["policy"]["models"],
    "evaluation_csv_matches_predictions": bool(np.allclose(ev["pd_final"], p_d, atol=1e-6)
                                               and np.allclose(ev["pd_comparison"], p_a, atol=1e-6)),
    "features_used_by_d": len(bundle["final_model"].named_steps["prep"].get_feature_names_out()),
}))
'''
result = subprocess.run([sys.executable, "-c", VERIFY_SCRIPT], capture_output=True, text=True, check=True)
verified = json.loads(result.stdout)
pd.DataFrame({"(d) Final model": verified["final_model_d"],
              "(a) Comparison model": verified["comparison_model_a"]})

,(d) Final model,(a) Comparison model
lower,0.15,0.11
upper,0.71,0.58
cost,467,409
recall,0.949,0.958
green,0.301,0.341
yellow,0.599,0.421
red,0.1,0.238
red_precision,0.86,0.819
good_auto_declined,14,43
defaults_auto_approved,17,14


In [8]:
# Expected test-set values from Steps 5 and 6
expected_d = {"lower": 0.15, "upper": 0.71, "cost": 467, "recall": 0.949, "green": 0.301,
              "yellow": 0.599, "red": 0.100, "red_precision": 0.860, "good_auto_declined": 14,
              "defaults_auto_approved": 17, "eod_age": 0.365, "eod_sex": 0.174,
              "good_51plus_auto_declined": "0 of 21"}
expected_a = {"lower": 0.11, "upper": 0.58, "cost": 409, "recall": 0.958, "green": 0.341,
              "yellow": 0.421, "red": 0.238, "red_precision": 0.819, "good_auto_declined": 43,
              "defaults_auto_approved": 14, "eod_age": 0.812, "eod_sex": 0.109,
              "good_51plus_auto_declined": "8 of 21"}

checks = []
for name, got, exp in [("(d)", verified["final_model_d"], expected_d),
                       ("(a)", verified["comparison_model_a"], expected_a)]:
    for k, v in exp.items():
        checks.append({"Model": name, "Metric": k, "Expected": v, "Reloaded": got[k], "Match": got[k] == v})
checks = pd.DataFrame(checks)
print("All values match:", bool(checks["Match"].all()))
print("policy.json matches the bundle:", verified["policy_json_matches_bundle"])
print("evaluation_data.csv matches the reloaded predictions:", verified["evaluation_csv_matches_predictions"])
assert checks["Match"].all() and verified["policy_json_matches_bundle"] and verified["evaluation_csv_matches_predictions"]
checks[~checks["Match"]] if not checks["Match"].all() else checks.groupby("Model")["Match"].agg(["sum", "count"])

All values match: True
policy.json matches the bundle: True
evaluation_data.csv matches the reloaded predictions: True


,sum,count
Model,,
(a),13,13
(d),13,13


**Result.** The saved files reproduce the results of Steps 5–7 exactly:

| Test set (reloaded in a separate process) | (d) Final model | (a) Comparison model |
|---|---|---|
| Boundaries | 0.15 / 0.71 | 0.11 / 0.58 |
| Cost (single-threshold logic) | 467 | 409 |
| Recall | 0.949 | 0.958 |
| Green / yellow / red | 30.1% / 59.9% / 10.0% | 34.1% / 42.1% / 23.8% |
| Red-zone precision | 86.0% | 81.9% |
| Good customers declined automatically | 14 | 43 |
| Defaults auto-approved | 17 | 14 |
| EOD age group / sex | 0.365 / 0.174 | 0.812 / 0.109 |
| Good customers aged 51+ declined automatically | 0 of 21 | 8 of 21 |

- All 26 checked values match the expected values from Steps 5 and 6.
- `policy.json` contains the same boundaries as the bundle, and `evaluation_data.csv` contains the same predictions as the reloaded models.
- The stored SHAP data gives exactly the same values as the `shap` library (difference 0.0), and the contributions add up to the model's log-odds.
- The final model uses 56 model columns (no `Sex` or `ForeignWorker` columns); the comparison model uses 60.

**How the dashboard will use the files.**
- *Single application / batch upload:* `final_model.predict_proba()` gives the default probability; the zone follows from the stored boundaries; the explanation is computed from `explain_data` (coefficient × deviation from the average applicant), with age labelled "Age (fairness correction)".
- *Bank view:* the target-precision slider looks up the upper boundary (or recomputes it from the out-of-fold predictions in `evaluation_data.csv` for any value), and cost, recall, zone shares and fairness metrics are recomputed on the test rows of the same file. The toggle switches between `pd_final` and `pd_comparison`.

**Limitations.**
- The pickled pipelines depend on the scikit-learn version (1.9.1, stored in the metadata and pinned in `requirements.txt`). Loading them with a different version may fail or give warnings.
- The models are trained on the 4,000 training applications only, so that the test results remain verifiable. A production model would be refit on all data and revalidated.
- `evaluation_data.csv` contains the sensitive attributes of the (synthetic) applicants for the fairness metrics. With real customer data, this file would need data-protection safeguards.
- Rating classes A–E are not part of the saved policy yet; they will be defined with the dashboard in Step 9.